# 🧠 Fase 2: Filtragem Baseada em Conteúdo (Content-Based Filtering)
Neste notebook, vamos desmistificar o "motor" do Sistema de Recomendação. 
Vamos construir a matemática passo a passo: desde a transformação de texto em números (TF-IDF) até a montagem do perfil do usuário com base no que ele curtiu (+) e rejeitou (-).


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Configuração visual
sns.set_theme(style="whitegrid")


## 1. Carregamento dos Dados
Vamos carregar nossa base de `postings.csv`. Para focar na qualidade do recomendador e não estourar a memória (já que o TF-IDF cria matrizes gigantescas), vamos filtrar apenas vagas ativas (vamos limitar a uma amostra limpa) e com informações essenciais preenchidas.


In [2]:
base_path = "data/raw" if os.path.exists("data/raw") else "."

# Carrega apenas as colunas necessárias para economizar RAM
colunas = ['job_id', 'title', 'skills_desc', 'formatted_experience_level', 'remote_allowed', 'applies', 'views']
df = pd.read_csv(f"{base_path}/postings.csv", usecols=lambda c: c in colunas)

# Limpeza e Tratamento
df = df.dropna(subset=['title'])

# Preenche nulos nas skills
df['skills_desc'] = df['skills_desc'].fillna('')
df['formatted_experience_level'] = df['formatted_experience_level'].fillna('')

# Calcula o CTR que definimos na Fase 1
df['applies'] = df['applies'].fillna(0)
df['views'] = df['views'].fillna(1) # evita divisão por zero
df['ctr'] = df['applies'] / df['views']
df['ctr'] = df['ctr'].clip(upper=1.0) # Limita candidaturas externas anômalas

# Flag de remoto
df['is_remote'] = df['remote_allowed'].fillna(0).astype(int)

# Pega uma amostra de 25 mil vagas para manter o processamento rápido neste laboratório
df = df.sample(n=min(25000, len(df)), random_state=42).reset_index(drop=True)
print(f"Base de laboratório pronta com {len(df)} vagas.")


Base de laboratório pronta com 25000 vagas.


## 2. A "Sopa de Palavras" (Metadados do Item)
Em recomendação baseada em conteúdo, precisamos representar o item (a vaga) como um documento de texto. 
Como o **título** é o atributo mais forte, daremos peso duplo a ele na concatenação.
$$ \text{Documento}_{i} = \text{Título} \times 2 + \text{Skills} + \text{Nível} $$


In [3]:
def build_item_string(row):
    # Peso 2 para o título para forçar a semântica principal
    title_str = (str(row['title']) + " ") * 2 
    skills_str = str(row['skills_desc'])
    level_str = str(row['formatted_experience_level']).replace(" ", "") # Junta "Entry level" para "Entrylevel"
    
    return f"{title_str} {skills_str} {level_str}".lower()

df['item_string'] = df.apply(build_item_string, axis=1)

# Vejamos como ficou o "documento" da primeira vaga:
print("Exemplo de Representação Vetorial de uma Vaga:")
print("-" * 50)
print(df[['title', 'item_string']].iloc[0]['item_string'])


Exemplo de Representação Vetorial de uma Vaga:
--------------------------------------------------
senior automation engineer - power systems senior automation engineer - power systems   mid-seniorlevel


## 3. Vetorização (TF-IDF)
A matemática entra aqui. O TF-IDF (*Term Frequency-Inverse Document Frequency*) vai punir palavras que aparecem em todas as vagas (como "and", "the", "work") e dar um score altíssimo para palavras raras e específicas (como "PyTorch", "Kubernetes", "B2B Sales").


In [4]:
# Configuramos n-gramas de (1,2) para pegar combinações como "data scientist" ou "machine learning"
tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=5,           # Ignora termos que aparecem em menos de 5 vagas
    max_features=5000,  # Limita aos 5000 termos mais relevantes do mercado
    stop_words='english'
)

# Transforma a coluna de texto na grande matriz matemática (Vagas x Termos)
tfidf_matrix = tfidf.fit_transform(df['item_string'])

print(f"Dimensão da Matriz TF-IDF: {tfidf_matrix.shape}")
print(f"(Temos {tfidf_matrix.shape[0]} vagas representadas num espaço de {tfidf_matrix.shape[1]} dimensões de palavras)")


Dimensão da Matriz TF-IDF: (25000, 5000)
(Temos 25000 vagas representadas num espaço de 5000 dimensões de palavras)


## 4. O Perfil do Usuário (Matemática Pura)
Um usuário não é nada além do histórico dele. Se ele gostou das vagas A e B, o perfil dele é a soma dos vetores de A e B. Se ele rejeitou a vaga C, nós subtraímos o vetor de C.
$$ \vec{u} = \alpha \sum_{i \in I^+} \vec{v}_i - \beta \sum_{j \in I^-} \vec{v}_j $$
Onde $\alpha$ é o peso do "Gostei" (ex: 1.0) e $\beta$ é o peso do "Não Gostei" (ex: 0.5).


In [5]:
def build_user_profile(positive_indices, negative_indices, matrix, alpha=1.0, beta=0.5):
    """
    Constrói o vetor numérico do perfil do usuário.
    """
    user_vector = np.zeros((1, matrix.shape[1]))
    
    # Soma os vetores positivos
    if positive_indices:
        positive_vectors = matrix[positive_indices]
        # Soma todos os vetores ao longo do eixo das vagas, multiplicado pelo peso alfa
        user_vector += alpha * np.asarray(positive_vectors.sum(axis=0))
        
    # Subtrai os vetores negativos
    if negative_indices:
        negative_vectors = matrix[negative_indices]
        user_vector -= beta * np.asarray(negative_vectors.sum(axis=0))
        
    # Normalizamos o vetor para evitar que usuários muito ativos tenham scores muito maiores
    # que usuários novos (apenas direção importa, não magnitude)
    if np.linalg.norm(user_vector) > 0:
        user_vector = user_vector / np.linalg.norm(user_vector)
        
    return user_vector


## 5. Função de Recomendação (Similaridade do Cosseno + CTR)
Para cada vaga, vamos calcular o Cosseno do Ângulo entre o Vetor do Usuário e o Vetor da Vaga. 
Quanto mais próximo de 1, mais alinhado. Depois, multiplicamos pelo Bônus de CTR, aplicando a **Prudência Epistemológica** (valorizamos vagas com alta conversão empiricamente provada).


In [6]:
def recommend_jobs(user_profile, matrix, df_data, top_n=10, ctr_weight=0.2, remote_only=False):
    # Calcula a similaridade do cosseno entre o usuário e TODAS as vagas (retorna matriz 1 x N)
    cosine_sim = cosine_similarity(user_profile, matrix).flatten()
    
    # Cria um DataFrame de resultados
    results = df_data[['job_id', 'title', 'formatted_experience_level', 'is_remote', 'ctr']].copy()
    results['similarity'] = cosine_sim
    
    # Modulador de CTR (Conforme nossa hipótese do EDA)
    # Vagas muito populares ganham um bônus no score de até ctr_weight
    results['final_score'] = results['similarity'] * (1.0 + (ctr_weight * results['ctr']))
    
    # Filtro Rígido
    if remote_only:
        results = results[results['is_remote'] == 1]
        
    # Ordena pelo score final
    recommended = results.sort_values(by='final_score', ascending=False)
    
    return recommended.head(top_n)


## 6. Testando as Recomendações (Personas)
Vamos simular as interações para provar que a matemática funciona.


In [8]:
# Buscando índices reais na base de dados para usarmos como "Gostei" / "Não Gostei"
# Persona A: Cientista de Dados que odeia vagas de marketing
ds_vagas = df[df['title'].str.contains('Data Scientist', case=False, na=False)].index.tolist()[:3]
mkt_vagas = df[df['title'].str.contains('Marketing', case=False, na=False)].index.tolist()[:2]

print("Vagas curtidas pela Persona A:")
display(df.loc[ds_vagas, ['title', 'formatted_experience_level']])

print("\n Vagas REJEITADAS pela Persona A:")
display(df.loc[mkt_vagas, ['title', 'formatted_experience_level']])


Vagas curtidas pela Persona A:


,title,formatted_experience_level
1502,"Senior Data Scientist, Experimentation & Perso...",Mid-Senior level
1844,Senior Data Scientist,Mid-Senior level
3576,Senior Data Scientist,Mid-Senior level



 Vagas REJEITADAS pela Persona A:


,title,formatted_experience_level
93,Direct-to-Consumer Marketing & Distribution St...,Internship
175,Senior Channel Marketing Manager or Channel Ma...,Mid-Senior level


In [9]:
# Gerando o Perfil
persona_a_vector = build_user_profile(
    positive_indices=ds_vagas, 
    negative_indices=mkt_vagas, 
    matrix=tfidf_matrix,
    alpha=1.0, 
    beta=1.0 # Penalidade severa
)

# Recomendação sem filtro de remoto
recs_a = recommend_jobs(persona_a_vector, tfidf_matrix, df, top_n=5)
print("🎯 TOP 5 RECOMENDAÇÕES PARA A PERSONA A (Data Scientist):")
display(recs_a)


🎯 TOP 5 RECOMENDAÇÕES PARA A PERSONA A (Data Scientist):


,job_id,title,formatted_experience_level,is_remote,ctr,similarity,final_score
8343,3901697459,Senior Data Scientist,Mid-Senior level,0,0.396552,0.884142,0.954263
1844,3900947017,Senior Data Scientist,Mid-Senior level,1,0.222222,0.884142,0.923437
16987,3887985726,Senior Data Scientist,Mid-Senior level,1,0.000000,0.884142,0.884142
3576,3905826251,Senior Data Scientist,Mid-Senior level,0,0.000000,0.884142,0.884142
1502,3894285749,"Senior Data Scientist, Experimentation & Perso...",Mid-Senior level,0,0.000000,0.858971,0.858971


In [10]:
# Persona B: RH buscando vagas de recrutamento remoto
hr_vagas = df[df['title'].str.contains('Recruiter|Human Resources', case=False, na=False)].index.tolist()[:3]
dev_vagas = df[df['title'].str.contains('Software|Developer', case=False, na=False)].index.tolist()[:2]

persona_b_vector = build_user_profile(
    positive_indices=hr_vagas, 
    negative_indices=dev_vagas, 
    matrix=tfidf_matrix
)

# Recomendação exigindo filtro remoto
recs_b = recommend_jobs(persona_b_vector, tfidf_matrix, df, top_n=5, remote_only=True)
print("🎯 TOP 5 RECOMENDAÇÕES PARA A PERSONA B (Recrutador / Apenas Remoto):")
display(recs_b)


🎯 TOP 5 RECOMENDAÇÕES PARA A PERSONA B (Recrutador / Apenas Remoto):


,job_id,title,formatted_experience_level,is_remote,ctr,similarity,final_score
1836,3905237863,Human Resources Administrator,Associate,1,0.571429,0.706611,0.787367
20321,3903487554,Human Resources Representative,Mid-Senior level,1,0.392857,0.726687,0.783784
2738,3899526653,Human Resources Intern,,1,0.254902,0.745673,0.783687
9823,3900080433,Human Resources Intern,,1,0.181818,0.745673,0.772788
19497,3891084006,Human Resources Manager,Mid-Senior level,1,0.142857,0.654407,0.673105
